In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

## Visualizations of SAM3 on LVIS

This notebook shows visualizations of how SAM3 performs on LVIS image data. Run the first cell once for setup, and rerun the second cell to view random examples from the val dataset. This only plots predictions with >=.5 confidence. Run the 3rd cell to save visualizations of the entirety of the results. Change paths as needed. 

# <a target="_blank" href="https://colab.research.google.com/github/facebookresearch/sam3/blob/main/notebooks/sam3_image_predictor_example.ipynb">
#   <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
# </a>

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

import sam3
from PIL import Image
from sam3 import build_sam3_image_model
from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results, generate_colors, plot_mask
import json
import matplotlib.patches as patches
from pycocotools import mask as maskUtils
import random
from collections import defaultdict

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")
json_path = f"{sam3_root}/sam3/train/configs/lvis/cse_densepose_lvis_v1_ds2_val_v1_filtered.json"
results_path = f"{sam3_root}/log_dir/dumps/lvis/}}/coco_predictions_segm.json"
img_folder = f"{sam3_root}/../lvis/combined2017"
output_plots_folder = f"{sam3_root}/../lvis/sam3baseresults"

with open(results_path, "r") as f:
    data = json.load(f)

img_to_results = defaultdict(list)
for result in data:
    img_to_results[result["image_id"]].append(result)


def plot_results_for_img(img_id):
     # --- LOAD IMAGE ---
    image_filename = f"{img_id:012d}.jpg"
    image_path = os.path.join(img_folder, image_filename)
    img = np.array(Image.open(image_path))
    
    
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(img)
    COLORS = generate_colors(n_colors=128, n_samples=5000)
    
    i=0
    for result in img_to_results[img_id]:
        if result["score"] < .5:
            continue
        color = COLORS[i % len(COLORS)]
        i += 1
        # Get the random result
        image_id = result["image_id"]
        score = result.get("score", 0)
        category_id = result.get("category_id", "N/A")
    
        segmentation_present = "segmentation" in result
        # Decode and plot segmentation mask 
        if segmentation_present:
            segm = result["segmentation"]
            # segm can be either RLE dict or a list of polygons
            if isinstance(segm, dict) and "counts" in segm:
                # RLE format
                mask = maskUtils.decode(segm)
                plot_mask(mask, color=color)  # overlay mask with transparency
            # elif isinstance(segm, list):
            #     # Polygon format
            #     for poly in segm:
            #         poly = np.array(poly).reshape(-1, 2)
            #         patch = patches.Polygon(poly, linewidth=2, edgecolor='red', facecolor='red', alpha=0.5)
            #         ax.add_patch(patch)
         # Draw bounding box if available
        if "bbox" in result:
            x, y, w, h = result["bbox"]
            if segmentation_present:
                x *= result["segmentation"]["size"][1]
                w *= result["segmentation"]["size"][1]
                y *= result["segmentation"]["size"][0]
                h *= result["segmentation"]["size"][0]
            rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='green', facecolor='none')
            ax.add_patch(rect)
            ax.text(x, y - 10, f"CatID: {category_id}, Score: {score:.2f}",
                    color='green', fontsize=12, backgroundcolor=(0,0,0,0))

In [ ]:
rand_img = random.choice(list(img_to_results.keys()))

plot_results_for_img(rand_img)
plt.axis('off')
plt.show()
    

Run the following cell to process the entirety of the results json and save visualizations.

In [ ]:
for img in img_to_results.keys():
    plot_results_for_img(img)
    plt.savefig(os.path.join(output_plots_folder, f"{img:012d}.jpg"))
    plt.clf()